# **Ejercicio 4 - Proyecto Usos IA**

En este notebook especificamos el código que redacta el comentario de mercado del Diario de Bolsa, utilizando el modelo LLM GPT-4o de OpenAI.

Evidentemente, hemos obtenido el código, al que ha hecho falta hacer algunas modificaciones, de la inteligencia artificial, como describimos en el fichero de texto del proyecto. Además de DeepSeek para el código digamos base, el corrector de IA de Colab ha hecho contribuciones importantes.

A lo largo del notebook hiremos haciendo los comentarios pertinentes, que complementan al fichero de texto.


Para empezar instalamos las librerías necesarias, comenzando por yfinance, que es de donde extraeremos los datos. Como se menciona en el texto, en el futuro usaremos la API de Bloomberg.

In [ ]:
! pip install yfinance

DeepSeek nos recomienda instalar la librería de newspaper3k.

In [ ]:
!pip install newspaper3k

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 50.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 10.0 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=bad4dd5438500a721118360da63e595e1ed139cb0ccf67376ab1f329df395ebc
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=8a2b7e1d6666426a0d4d4c6900e461556804fefd1abcf12712f8bc09e5f0f05a
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f1

Ahora cargamos las librerías que necesitamos. en el caso de OpenAI nos aseguramos que cargamos la version 1.0.0, mientras que en el caso de newspaper, Colab nos recomienda instalar una librería adicional, lxml_html_clean.

In [ ]:
from openai import OpenAI

In [ ]:
!pip install lxml_html_clean
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from newspaper import Article
# import openai
from googlesearch import search
import json
from typing import Dict, List, Optional

Casi mejor quitar los warnings.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

# Instrucciones de uso

El proceso es bastante simple: basta con especificar la fecha para la que se quiere el diario en la variable dia_diario de la función main() en la segunda celda, y ejecutar, tras haber ejecutado las celdas anteriores.

Hay que facilitar, también dentro de la funcion main(), un path para el fichero de Excel. El presente en el código es el link a mi Colab.

**La clave de OpenAI se ha borrado por motivos de seguridad.**

He pensado que es más útil el dejar los prints generados durante el proceso, en especial el del prompt completo, para dar más claridad.

# Manejo de errores

Si algo hay que agradecer a DeepSeek y Colab es el empeño que han puesto en el manejo de errores. Casi todas las funciones tienen rutinas de manejo de errores, algo en lo que, reconozco, no soy personalmente muy bueno en mi código. El ejemplo más claro es el del fallback en el caso de que GTP-4o falle, se aplicaría otro LLM (3.5).

# Descripción del código

Pasamos a mostrar el código de la solución. Básicamente se divide en dos celdas:

- **Primera Celda**: Clases con sus métodos
- **Segunda Celda**: Función main()

# Primera Celda

La primera celda es la que tenemos a continuación. Se compone de dos clases:

- **Clase MarketDataScraper**: Recolecta (scrapea) los datos de los mercados.
- **Clase DiarioGenerator**: Genera el comentario de mercado del Diario de Bolsa

A continuación explicamos el contenido de cada clase.

## **Clase MarketDataScraper**

La clase MarketDataScraper obtiene, a partir principalmente de la librería de yfinance, los datos de mercados con los que luego se forma el prompt. Tambien se utilizan otras páginas web para las noticias económicas. Esencialmente es un scrapping de la web, que NO utiliza la IA.

El método get_market_data centraliza la recopilación de los datos de mercado, a través de los siguientes métodos:

- **_get_previous_business_day**: Obtiene la fecha para hacer el scrapping.

- **_get_index_data**: Obtiene las cotizaciones de los principales índices bursátiles, así como sus variaciones desde comienzo de año.
- **_get_sector_data**: Consigue las cotizaciones de sectores económicos a nivel europeo, y sus variaciones.
- **_get_ibex_stocks**: Busca las cotizaciones de los componentes del Ibex 35, así como sus variaciones.
- **_get_commodity_data**: Consigue los precios de las principales materias primas, y sus variaciones.
- **_get_forex_data**: Obtiene los datos de los mercados cambiarios (monedas).
- **_get_economic_news**: Consigue las principales noticias económicas a partir de varias fuentes, llevando a cabo un scrapping clásico.
- **_get_macro_calendar**: En este caso añadimos los principales eventos económicos del día y la víspera manualmente, aunque cuando usemos Bloomberg lo podremos hacer de forma automática.


## **Clase DiarioGenerator**

En esta clase utilizamos el LLM para que cree el comentario de mercado. En el método __init__ creamos el cliente de OpenAI. Posteriormente se encuentran los siguientes métodos:

- **_load_historical_examples**: Aquí cargamos el fichero Excel que contiene los ejemplos de comentarios anteriores.

- **_find_similar_examples**: Se seleccionan los comentarios antiguos con más similitud temporal al que estamos buscando. Esta función fue aportada, sin que ni por asomo se le pidiese, por DeepSeek.
- **_format_market_data_for_prompt**: Se formatean los datos de mercados para ser aportados al prompt.
- **_format_news_for_prompt**: Se formatean las noticias, si las hubiese, para ser aportadas al prompt.
- **generate_diario**: Este método dirige la generación del comentario, dando la orden de recolectar los datos, formatearlos, escribir el prompt, pasárselo al LLM, y parsear el comentario. Por alguna razón DeepSeek le colocó en este lugar, no al final de la clase.
- **_create_prompt**: Este método crea el prompt una vex formateados los datos, describiendo para el LLM la estructura del comentario de mercado.
- **_generate_with_gpt**: En este método es donde finalmente el LLM genera el comentario de mercado a partir del prompt que hemos diseñado anteriormente.
- **_parse_gpt_response**: Parseamos ligeramente el comentario obtenido del LLM, más que nada para asegurarnos que contiene tres párrafos.


Una vez que conocemos el guion de los métodos, la estructura del código es relativamente clara. Con todo, no deja de ser una mezcla entre el código original de DeepSeek, y los cambios efectuados por este LLM en su chat, y los llevados a cabo a partir de los consejos de Colab.

Algunos de ellos son los siguientes:

- **LLM**: El único cambio que afectó al LLM fue el pasar del código original, que no incluía un cliente, básicamente la versión anterior a la 1.0.0, a ponerlo al día introduciendo el cliente. Esto fue sugerido por DeepSeek a través de su chat.

- **Resto**: El resto de cambios fueron relacionados con los formatos de los datos. Por ejemplo, DeepSeek recomendó usar el método .ticker con la librería yfinance, mientras que Colab adaptó el código para seguir usando el original, .download. Yo modifiqué algo el formato del prompt, lo que simplificó bastante el parserado del comentario tras generarlo el LLM. Colab pienso que se excedió en lo complicado de la extracción de las cotizaciones de las accione, aunque decidí mantener su código.

Por otro lado, he respetado los comentarios originales de DeepSeek, que en el caso de ser introducidos con un '#' he puesto DS al principio. Los comentarios en inglés fueron añadidos por Colab.

In [ ]:
class MarketDataScraper:
    """Recolector de datos de mercado de fuentes públicas"""

    def __init__(self):
        self.user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        self.headers = {'User-Agent': self.user_agent}

    def get_market_data(self, fecha: datetime) -> Dict:
        """Obtener todos los datos de mercado necesarios"""

        # DS - Determinar fecha para yfinance (último día hábil)
        fecha_yfinance = self._get_previous_business_day(fecha)

        datos = {
            'fecha': fecha,
            'fecha_datos': fecha_yfinance,
            'indices': self._get_index_data(fecha_yfinance),
            'sectores': self._get_sector_data(fecha_yfinance),
            'ibex_stocks': self._get_ibex_stocks(fecha_yfinance),
            'commodities': self._get_commodity_data(fecha_yfinance),
            'forex': self._get_forex_data(fecha_yfinance),
            'news': self._get_economic_news(fecha),
            'macro_data': self._get_macro_calendar(fecha)
        }

        return datos

    def _get_previous_business_day(self, fecha: datetime) -> datetime:
        """Obtener el último día hábil"""

        # DS - Lógica para encontrar día hábil anterior
        offset = max(1, (fecha.weekday() + 6) % 7 - 3)

        return fecha - timedelta(days=offset)

    def _get_index_data(self, fecha: datetime) -> Dict:
        """Versión simplificada y robusta para obtener datos de índices"""

        tickers = {
            'EuroStoxx50': '^STOXX50E',
            'Nasdaq100': '^NDX',
            'S&P500': '^GSPC',
            'DAX': '^GDAXI',
            'CAC40': '^FCHI',
            'FTSE100': '^FTSE',
            'IBEX35': '^IBEX'
        }

        index_data = {}

        for nombre, ticker in tickers.items():
            try:
                print(f"  📊 Procesando {nombre}...")

                # DS - Usar yf.Ticker para más control
                ticker_obj = yf.Ticker(ticker)

                # DS - Obtener datos históricos
                end_date = (fecha + timedelta(days=1)).strftime('%Y-%m-%d')
                #start_date = (fecha - timedelta(days=353)).strftime('%Y-%m-%d')
                start_date = '2025-01-01'

                hist = ticker_obj.history(
                    start=start_date,
                    end=end_date,
                    interval='1d'
                )

                if hist.empty:
                    print(f"    ⚠️  Sin datos para {nombre}")
                    index_data[nombre] = None
                    continue

                # DS - Verificar que tenemos suficientes datos
                if len(hist) < 2:
                    print(f"    ⚠️  Pocos datos para {nombre}: {len(hist)} días")
                    index_data[nombre] = None
                    continue

                # DS - Obtener precios del día objetivo (último día disponible)
                current_close = float(hist['Close'].iloc[-1])
                previous_close = float(hist['Close'].iloc[-2])
                ytd_close = float(hist['Close'].iloc[0])  # Primer día del periodo

                # DS - Calcular variaciones
                daily_change = ((current_close - previous_close) / previous_close * 100) if previous_close != 0 else 0
                ytd_change = ((current_close - ytd_close) / ytd_close * 100) if ytd_close != 0 else 0

                index_data[nombre] = {
                    'precio': round(current_close, 2),
                    'variacion_dia': round(daily_change, 2),
                    'variacion_ano': round(ytd_change, 2),
                    'moneda': 'EUR' if nombre in ['EuroStoxx50', 'IBEX35'] else 'USD'
                }

                print(f"    ✅ {nombre}: {current_close:.2f} ({daily_change:+.2f}%)")

            except Exception as e:
                print(f"    ❌ Error {nombre}: {str(e)[:80]}")
                index_data[nombre] = {
                    'precio': None,
                    'variacion_dia': None,
                    'variacion_ano': None,
                    'moneda': 'EUR' if nombre in ['EuroStoxx50', 'IBEX35'] else 'USD'
                }

        return index_data

    def _get_sector_data(self, fecha: datetime) -> Dict:
        """Obtener rendimiento por sectores usando ETFs sectoriales (versión reparada)"""

        # DS - ETFs europeos por sectores (listados en Euronext)
        sector_etfs = {
            'Bancos': 'EXX1.DE',  # iShares STOXX Europe 600 Banks
            'Tecnología': 'SX8.DE',  # iShares STOXX Europe 600 Technology
            'Utilities': 'EXX5.DE',  # iShares STOXX Europe 600 Utilities
            'Energía': 'EXX3.DE',  # iShares STOXX Europe 600 Oil & Gas
            'Automoción': 'EXX7.DE',  # iShares STOXX Europe 600 Automobiles & Parts
            'Farmacia': 'EXXP.DE',  # iShares STOXX Europe 600 Chemicals
            'Telecoms': 'EXS1.DE',  # iShares STOXX Europe 600 Telecommunications
            'Construcción': 'EXXH.DE',  # iShares STOXX Europe 600 Construction & Materials
        }

        sector_data = {}

        for sector, ticker in sector_etfs.items():
            try:
                print(f"  📊 Procesando sector {sector} (ticker: {ticker})...")

                # DS - Usar yf.Ticker para más control
                ticker_obj = yf.Ticker(ticker)

                # DS - Obtener datos históricos
                end_date = (fecha + timedelta(days=1)).strftime('%Y-%m-%d')
                #start_date = (fecha - timedelta(days=30)).strftime('%Y-%m-%d')
                start_date = '2025-01-01'

                hist = ticker_obj.history(
                    start=start_date,
                    end=end_date,
                    interval='1d'
                )

                if hist.empty:
                    print(f"    ⚠️  Sin datos para {sector}")
                    sector_data[sector] = {
                        'precio': None,
                        'variacion_dia': None,
                        'variacion_ano': None,
                        'moneda': 'EUR'
                    }
                    continue

                # Ds - Verificar que tenemos suficientes datos
                if len(hist) < 2:
                    print(f"    ⚠️  Pocos datos para {sector}: {len(hist)} días")
                    sector_data[sector] = {
                        'precio': None,
                        'variacion_dia': None,
                        'variacion_ano': None,
                        'moneda': 'EUR'
                    }
                    continue

                # DS - Obtener precios del día objetivo (último día disponible)
                current_close_val = hist['Close'].iloc[-1]
                previous_close_val = hist['Close'].iloc[-2]
                ytd_close_val = hist['Close'].iloc[0]

                # Ensure values are scalar floats
                current_close = float(current_close_val.item()) if isinstance(current_close_val, pd.Series) else float(current_close_val)
                previous_close = float(previous_close_val.item()) if isinstance(previous_close_val, pd.Series) else float(previous_close_val)
                ytd_close = float(ytd_close_val.item()) if isinstance(ytd_close_val, pd.Series) else float(ytd_close_val)


                # DS - Calcular variaciones
                daily_change = ((current_close - previous_close) / previous_close * 100) if previous_close != 0 else 0
                ytd_change = ((current_close - ytd_close) / ytd_close * 100) if ytd_close != 0 else 0

                sector_data[sector] = {
                    'precio': round(current_close, 2),
                    'variacion_dia': round(daily_change, 2),
                    'variacion_ano': round(ytd_change, 2),
                    'moneda': 'EUR'
                }

                print(f"    ✅ {sector}: {current_close:.2f} ({daily_change:+.2f}%)")

            except Exception as e:
                print(f"    ❌ Error {sector}: {str(e)[:80]}")
                sector_data[sector] = {
                    'precio': None,
                    'variacion_dia': None,
                    'variacion_ano': None,
                    'moneda': 'EUR'
                }

        # DS - Ordenar por rendimiento usando 'variacion_dia' y manejando valores None
        return dict(sorted(sector_data.items(),
                           key=lambda item: item[1]['variacion_dia'] if item[1] and item[1]['variacion_dia'] is not None else -999999, # Use a very small number for None to push them to the end when reverse=True
                           reverse=True))


    def _get_ibex_stocks(self, fecha: datetime) -> Dict:
        """Obtener datos de componentes del IBEX 35"""

        # DS - Lista de componentes principales del IBEX con sus tickers Yahoo Finance
        ibex_components = {
            'Santander': 'SAN.MC',
            'BBVA': 'BBVA.MC',
            'Telefónica': 'TEF.MC',
            'Inditex': 'ITX.MC',
            'Iberdrola': 'IBE.MC',
            'Repsol': 'REP.MC',
            'Ferrovial': 'FER.MC',
            'ACS': 'ACS.MC',
            'Mapfre': 'MAP.MC',
            'Grifols': 'GRF.MC',
            'Aena': 'AENA.MC',
            'IAG': 'IAG.MC',
            'Indra': 'IDR.MC',
            'Sabadell': 'SAB.MC',
            'CaixaBank': 'CABK.MC',
            'Unicaja': 'UNI.MC',
            'Rovi': 'ROVI.MC',
            'Colonial': 'COL.MC',
            'Merlin': 'MRL.MC',
            'Solaria': 'SLR.MC',
            'Fluidra': 'FDR.MC',
            'Naturgy': 'NTGY.MC',
            'Puig': 'PUI.MC',
            'Endesa': 'ELE.MC',
            'Redeia': 'RED.MC',
            'Enagás': 'ENG.MC',
            'Cellnex': 'CLNX.MC',
            'Acciona': 'ANA.MC',
            'Acciona Energía': 'ANE.MC',
            'Amadeus': 'AMS.MC',
            'Acerinox': 'ACX.MC'
        }

        stocks_data = {}

        for nombre, ticker in list(ibex_components.items()):  # Limitar para no sobrecargar
            try:
                data = yf.download(ticker, start=fecha - timedelta(days=2), end=fecha + timedelta(days=1), progress=False)

                if len(data) >= 2:
                    precio_actual_val = data['Close'].iloc[-1]
                    precio_anterior_val = data['Close'].iloc[-2]
                    volumen_val = data['Volume'].iloc[-1]

                    # Ensure values are scalar floats/integers, handling potential Series return from .iloc
                    precio_actual = float(precio_actual_val.item()) if isinstance(precio_actual_val, pd.Series) and not precio_actual_val.empty else float(precio_actual_val) if not pd.isna(precio_actual_val) else 0.0
                    precio_anterior = float(precio_anterior_val.item()) if isinstance(precio_anterior_val, pd.Series) and not precio_anterior_val.empty else float(precio_anterior_val) if not pd.isna(precio_anterior_val) else 0.0

                    # Calculate variation
                    variacion = ((precio_actual - precio_anterior) / precio_anterior) * 100 if precio_anterior != 0 else 0.0

                    # Ensure variacion is scalar float before storing (and rounding)
                    if isinstance(variacion, pd.Series):
                        if not variacion.empty:
                            variacion = float(variacion.iloc[0])
                        else:
                            variacion = float('nan') # Handle empty Series as NaN
                    elif not isinstance(variacion, (float, int)):
                        try:
                            variacion = float(variacion)
                        except (TypeError, ValueError): # Catch cases where conversion to float fails
                            variacion = float('nan')

                    # Robustly handle volumen, converting to int and handling NaN
                    if isinstance(volumen_val, pd.Series):
                        if not volumen_val.empty:
                            volumen = int(volumen_val.iloc[0]) if not pd.isna(volumen_val.iloc[0]) else 0
                        else:
                            volumen = 0
                    elif pd.isna(volumen_val):
                        volumen = 0
                    else:
                        volumen = int(volumen_val)

                    # Store values, handling potential NaN for variation before rounding
                    final_variacion = round(variacion, 2) if not pd.isna(variacion) else 0.0

                    stocks_data[nombre] = {
                        'precio': round(precio_actual, 2),
                        'variacion': final_variacion,
                        'volumen': volumen
                    }

                    # print(stocks_data)

            except Exception as e:
                print(f"Error {nombre}: {e}")
                continue

        # DS - Separar ganadores y perdedores
        # Ensure 'variacion' is always a comparable scalar for sorting

        print(stocks_data)

        ganadores = {k: v for k, v in sorted(stocks_data.items(),
                     key=lambda x: x[1]['variacion'], reverse=True)[:5]}

        perdedores = {k: v for k, v in sorted(stocks_data.items(),
                      key=lambda x: x[1]['variacion'])[:5]}

        return {
            'ganadores': ganadores,
            'perdedores': perdedores,
            'total': stocks_data
        }

    def _get_commodity_data(self, fecha: datetime) -> Dict:
        """Obtener precios de commodities"""

        commodities = {
            'Brent': 'BZ=F',      # Brent Crude
            'WTI': 'CL=F',        # WTI Crude
            'Oro': 'GC=F',        # Gold
            'Cobre': 'HG=F',      # Copper
            'Gas Natural': 'NG=F', # Natural Gas
            'Trigo': 'ZW=F',      # Wheat
        }

        commodity_data = {}

        for nombre, ticker in commodities.items():
            try:
                data = yf.download(ticker, start=fecha - timedelta(days=2),
                                 end=fecha + timedelta(days=1), progress=False)

                if len(data) > 0:
                    precio_val = data['Close'].iloc[-1]
                    # Ensure precio is a scalar float
                    precio = float(precio_val.item()) if isinstance(precio_val, pd.Series) else float(precio_val)

                    commodity_data[nombre] = {
                        'precio': round(precio, 2),
                        'moneda': 'USD',
                        'unidad': 'barril' if 'brent' in nombre.lower() else 'onza' if 'oro' in nombre.lower() else 'tonelada'
                    }
            except Exception as e:
                print(f"Error {nombre} commodity: {e}")
                continue

        return commodity_data

    def _get_forex_data(self, fecha: datetime) -> Dict:
        """Obtener tipos de cambio"""

        forex_pairs = {
            'EUR/USD': 'EURUSD=X',
            'EUR/GBP': 'EURGBP=X',
            'EUR/JPY': 'EURJPY=X',
            'USD/JPY': 'USDJPY=X',
            'GBP/USD': 'GBPUSD=X'
        }

        forex_data = {}

        for pair, ticker in forex_pairs.items():
            try:
                data = yf.download(ticker, start=fecha - timedelta(days=2),
                                 end=fecha + timedelta(days=1), progress=False)

                if len(data) > 0:
                    precio = data['Close'].iloc[-1]
                    forex_data[pair] = round(precio, 4)
            except:
                continue

        return forex_data

    def _get_economic_news(self, fecha: datetime) -> List[Dict]:
        """Obtener noticias económicas del día"""

        # DS - Fuentes de noticias económicas
        news_sources = [
            #'https://www.reuters.com/business',
            #'https://www.bloomberg.com/europe',
            #'https://www.ft.com',
            'https://www.elpais.com/economia',
            'https://www.expansion.com',
            'https://cincodias.elpais.com/'
        ]

        news_items = []

        for source in news_sources[:3]:  # DS - Limitar a 3 fuentes por tiempo
            try:
                # DS - Buscar noticias del día específico
                query = f"site:{source} {fecha.strftime('%Y-%m-%d')} economía, empresas, mercados"
                search_results = list(search(query, start=0, stop=5, lang='es'))

                for url in search_results:
                    try:
                        article = Article(url, headers=self.headers)
                        article.download()
                        article.parse()

                        if article.publish_date and article.publish_date.date() == fecha.date():
                            news_items.append({
                                'titulo': article.title,
                                'resumen': article.text[:300] + '...',
                                'fuente': source,
                                'fecha': article.publish_date,
                                'url': url
                            })
                    except:
                        continue

            except Exception as e:
                print(f"Error scraping {source}: {e}")
                continue

        return news_items[:10]  # DS - Limitar a 10 noticias

    def _get_macro_calendar(self, fecha: datetime) -> Dict:
        """Obtener calendario macroeconómico"""

        # DS - Para este ejemplo, simular datos macro
        # DS - En producción, usarías APIs como Trading Economics, Investing.com, etc.

        macro_events = {
            'hoy': [
                {'hora': '16:00', 'pais': 'EZ', 'indicador': 'Confianza del Consumidor', 'consenso': '-18.0'},
                {'hora': '08:00', 'pais': 'UK', 'indicador': 'Ventas Minoristas YoY', 'consenso': '1.9%'},
                {'hora': '16:00', 'pais': 'EEUU', 'indicador': 'U. Michigan Dic.', 'consenso': 'No disponible'},
            ],
            'ayer': [
                {'indicador': 'US CPI', 'resultado': '2.7%', 'consenso': '3.0%'},
                {'indicador': 'Petición subsidios de desempleo', 'resultado': '224.000', 'consenso': '225.000'},
            ]
        }

        return macro_events


class DiarioGenerator:
    """Generador de Diario de Bolsa usando LLM y datos históricos"""

    def __init__(self, openai_api_key: str, excel_path: str):
        # self.openai_api_key = openai_api_key
        # openai.api_key = openai_api_key

        self.client = OpenAI(api_key=openai_api_key)

        # DS - Cargar ejemplos históricos
        self.historical_data = self._load_historical_examples(excel_path)
        self.scraper = MarketDataScraper()

    def _load_historical_examples(self, excel_path: str):
        """Cargar y procesar ejemplos históricos"""
        df = pd.read_excel(excel_path)

        # DS - Convertir a diccionario por fecha
        examples = {}
        for _, row in df.iterrows():
            fecha = pd.to_datetime(row['Dia_Publicación']).date()
            examples[fecha] = {
                'Parrafo_1': row['Parrafo_1'],
                'Parrafo_2': row['Parrafo_2'],
                'Parrafo_3': row['Parrafo_3']
            }

        return examples

    def _find_similar_examples(self, target_date: datetime, n_examples: int = 3):
        """Encontrar ejemplos históricos similares"""

        # DS - Buscar por día de la semana y época del año
        target_weekday = target_date.weekday()
        target_month = target_date.month

        similar = []

        for fecha_str, example in self.historical_data.items():
            fecha = pd.to_datetime(fecha_str).date()
            fecha_dt = datetime.combine(fecha, datetime.min.time())

            # DS - Calcular similitud (mismo día de semana, mes similar)
            similarity = 0
            if fecha_dt.weekday() == target_weekday:
                similarity += 2
            if abs(fecha_dt.month - target_month) <= 1:
                similarity += 1

            if similarity > 0:
                similar.append((similarity, example))

        # DS - Ordenar por similitud y tomar los mejores
        similar.sort(key=lambda x: x[0], reverse=True)
        return [ex for _, ex in similar[:n_examples]]

    def _format_market_data_for_prompt(self, market_data: Dict) -> str:
        """Formatear datos de mercado para el prompt"""

        formatted = f"""
        📅 FECHA DE LA SESIÓN: {market_data['fecha_datos'].strftime('%A, %d de %B de %Y')}

        📈 ÍNDICES PRINCIPALES:
        """

        # DS - Índices
        for idx_name, idx_data in market_data['indices'].items():
            if idx_data:
                precio_str = f"{idx_data['precio']:.2f}" if idx_data['precio'] is not None else "N/A"
                variacion_dia_str = f"{idx_data['variacion_dia']:+.2f}" if idx_data['variacion_dia'] is not None else "N/A"
                variacion_ano_str = f"{idx_data['variacion_ano']:+.2f}" if idx_data['variacion_ano'] is not None else "N/A"
                formatted += f"- {idx_name}: {precio_str} ({variacion_dia_str}% día, {variacion_ano_str}% año)\n"

        formatted += f"""

        🏦 SECTORES EUROPEOS (rendimiento del día):
        """

        print('Hola, indices formated!')

        # DS - Sectores
        for sector, rendimiento in list(market_data['sectores'].items())[:6]:
            if rendimiento and rendimiento['variacion_dia'] is not None:
                formatted += f"- {sector}: {rendimiento['variacion_dia']:+.2f}%\n"
            elif rendimiento:
                formatted += f"- {sector}: N/A %\n"
            else:
                formatted += f"- {sector}: Datos no disponibles\n"


        formatted += f"""

        🇪🇸 IBEX 35 - MEJORES Y PEORES VALORES:

        GANADORES:
        """

        # DS - Ganadores IBEX
        for stock, data in market_data['ibex_stocks']['ganadores'].items():
            precio_str = f"{data['precio']:.2f}" if data['precio'] is not None else "N/A"
            variacion_str = f"{data['variacion']:+.2f}" if data['variacion'] is not None else "N/A"
            formatted += f"- {stock}: {precio_str}€ ({variacion_str}%)\n"

        print('Hola, ganadores Ibex formated!')

        formatted += f"""

        PERDEDORES:
        """



        # DS - Perdedores IBEX
        for stock, data in market_data['ibex_stocks']['perdedores'].items():
            precio_str = f"{data['precio']:.2f}" if data['precio'] is not None else "N/A"
            variacion_str = f"{data['variacion']:+.2f}" if data['variacion'] is not None else "N/A"
            formatted += f"- {stock}: {precio_str}€ ({variacion_str}%)\n"

        print('Hola, perdedores Ibex formated!')

        formatted += f"""

        ⚡ COMMODITIES:
        """

        # DS - Commodities
        for comm, data in market_data['commodities'].items():
            if data and data['precio'] is not None:
                # Ensure data['precio'] is a scalar float for formatting
                precio_scalar = float(data['precio'].item()) if isinstance(data['precio'], pd.Series) else float(data['precio'])
                formatted += f"- {comm}: {precio_scalar:.2f} {data['moneda']}/{data['unidad']}\n"
            else:
                formatted += f"- {comm}: Datos no disponibles\n"

        print('Hola, commodities formated!')


        formatted += f"""

        💱 FOREX:
        """

        # DS - Forex
        for pair, rate in market_data['forex'].items():
            if rate is not None:
                formatted += f"- {pair}: {rate}\n"
            else:
                formatted += f"- {pair}: N/A\n"

        print('Hola forex formated!')


        return formatted

    def _format_news_for_prompt(self, market_data: Dict) -> str:
        """Formatear noticias para el prompt"""

        formatted = "📰 NOTICIAS ECONÓMICAS DESTACADAS:\n\n"

        for i, news in enumerate(market_data['news'][:5], 1):
            formatted += f"{i}. {news['titulo']}\n"
            formatted += f"   {news['resumen'][:150]}...\n"
            formatted += f"   Fuente: {news['fuente']}\n\n"

        formatted += "📊 DATOS MACRO RECIENTES:\n\n"

        for macro in market_data['macro_data']['ayer']:
            formatted += f"- {macro['indicador']}: {macro['resultado']} (consenso: {macro['consenso']})\n"

        formatted += "\n📅 AGENDA MACRO DE HOY:\n\n"

        for event in market_data['macro_data']['hoy']:
            formatted += f"- {event['hora']} ({event['pais']}): {event['indicador']} (consenso: {event['consenso']})\n"

        return formatted

    def generate_diario(self, dia_diario: datetime) -> Dict:
        """Generar el diario completo para una fecha específica"""

        print(f"🔍 Obteniendo datos para {dia_diario.strftime('%Y-%m-%d')}...")

        # DS - 1. Obtener datos de mercado
        # Determinación de 'market_data'
        market_data = self.scraper.get_market_data(dia_diario)

        # DS - 2. Buscar ejemplos históricos similares
        similar_examples = self._find_similar_examples(dia_diario)

        # DS - 3. Preparar prompt
        prompt = self._create_prompt(market_data, similar_examples, dia_diario)

        # DS - 4. Generar con GPT
        print("🤖 Generando comentario con GPT-4o...")
        comentario = self._generate_with_gpt(prompt)

        print("🤖 Comentario generado:")
        print(comentario)
        print("-" * 50)

        # DS - 5. Parsear respuesta
        parsed = self._parse_gpt_response(comentario)

        print("🤖 Comentario parseado:")
        print(parsed)
        print("-" * 50)



        return {
            'fecha_publicacion': dia_diario,
            'fecha_datos': market_data['fecha_datos'],
            'comentario': parsed,
            'datos_brutos': market_data,
            'prompt_utilizado': prompt[:500] + "..."  # Guardar parte del prompt para debugging
        }

    def _create_prompt(self, market_data: Dict, similar_examples: List, dia_diario: datetime) -> str:
        """Crear prompt detallado para GPT"""

        # DS - Formatear ejemplos históricos
        examples_text = "📚 EJEMPLOS HISTÓRICOS SIMILARES:\n\n"
        for i, example in enumerate(similar_examples, 1):
            examples_text += f"Ejemplo {i}:\n"
            examples_text += f"Párrafo 1: {example['Parrafo_1'][:200]}...\n"
            examples_text += f"Párrafo 2: {example['Parrafo_2'][:200]}...\n"
            examples_text += f"Párrafo 3: {example['Parrafo_3'][:200]}...\n\n"

        # DS - Formatear datos actuales
        market_text = self._format_market_data_for_prompt(market_data)
        news_text = self._format_news_for_prompt(market_data)

        prompt = f"""
        # TAREA: REDACTAR COMENTARIO DE MERCADO PARA DIARIO DE BOLSA

        ## CONTEXTO HISTÓRICO
        {examples_text}

        ## DATOS ACTUALES DE LA SESIÓN
        {market_text}

        ## NOTICIAS Y AGENDA ECONÓMICA
        {news_text}

        ## INSTRUCCIONES ESPECÍFICAS

        REDACTAR UN COMENTARIO DE MERCADO CON TRES PÁRRAFOS:

        1. PRIMER PÁRRAFO (~150 palabras):
           - Comenzar con evaluación general de la sesión
           - Mencionar EuroStoxx 50 y su variación
           - Analizar sectores europeos más fuertes/débiles
           - Comparar con Wall Street (Nasdaq 100)
           - Incluir Brent y EUR/USD

        2. SEGUNDO PÁRRAFO (~150 palabras):
           - Contexto económico y geopolítico
           - Datos macro recientes importantes
           - Agenda macro del día
           - Noticias sobre tipos de interés (Fed, BCE)
           - Situación geopolítica relevante
           - Resultados empresariales destacados si los hay

        3. TERCER PÁRRAFO (~150 palabras):
           - Análisis específico del IBEX 35
           - Comparación rendimiento vs EuroStoxx (YTD)
           - Mencionar 3-4 valores que más subieron (con %)
           - Mencionar 3-4 valores que más bajaron (con %)
           - Perspectiva sectorial en España

        ## FORMATO REQUERIDO:
        - Idioma: Español
        - Estilo: Profesional pero accesible
        - Tono: Analítico pero no sensacionalista
        - Precisión: Usar datos exactos proporcionados
        - Extensión: 400-500 palabras total

        ## ESTRUCTURA DE RESPUESTA:
        [texto del PRIMER PÁRRAFO aquí]

        [texto del SEGUNDO PÁRRAFO aquí]

        [texto del TERCER PÁRRAFO aquí]
        """

        print(prompt)

        return prompt



    def _generate_with_gpt(self, prompt: str) -> str:
        """Generar texto usando GPT-4o (API v1.0.0+)"""

        try:
            # DS - La llamada cambia ligeramente:
            response = self.client.chat.completions.create(  # ✅ Nuevo método
                model="gpt-4o",  # Puedes usar "gpt-4-turbo" si lo prefieres
                messages=[
                    {
                        "role": "system",
                        "content": """Eres un analista senior de mercados con 15 años de experiencia.
                        Tu especialidad es redactar comentarios diarios de bolsa para medios financieros españoles.
                        Eres preciso, analítico, y mantienes un tono profesional pero accesible.
                        Siempre basas tus análisis en datos concretos y evitas especulaciones infundadas."""
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.7,
                max_tokens=1200,
                top_p=0.9,
                frequency_penalty=0.1,
                presence_penalty=0.1
            )

            # print('Contenido de GTP 4')
            # print(response.choices[0].message.content)

            # DS - El acceso al contenido también cambia:
            return response.choices[0].message.content  # ✅ Usar atributos (punto), no diccionario

        except Exception as e:
            print(f"Error con GPT: {e}")
            # (Opcional) Fallback a otro modelo si quieres mantenerlo
            try:
                response = self.client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7,
                    max_tokens=1000
                )
                return response.choices[0].message.content
            except:
                return "Error generando comentario."



    def _parse_gpt_response(self, response: str) -> Dict:
        """Parsear la respuesta de GPT en los tres párrafos"""

        # Ds - Dividir por dobles saltos de línea (párrafos)
        parrafos = [p.strip() for p in response.strip().split('\n\n') if p.strip()]

        # DS - Si hay menos de 3 párrafos, intentar con saltos simples
        if len(parrafos) < 3:
            parrafos = [p.strip() for p in response.strip().split('\n') if p.strip()]

        # DS - Asegurar 3 elementos
        while len(parrafos) < 3:
            parrafos.append("")

        # DS - Limitar a 3 párrafos (por si hay más)
        parrafos = parrafos[:3]

        return {
            'parrafo_1': parrafos[0],
            'parrafo_2': parrafos[1],
            'parrafo_3': parrafos[2],
            'texto_completo': response
        }


# Segunda Celda

En esta segunda celda simplemente se contiene la función main(), donde se almacenan la clave de OpenAI, así como el día para el que queremos el comentario de mercado (dia_diario).

También se activa el proceso de su redacción, generando un objeto de la clase DiarioGenerador, al que se aplica el método generate_diario.

Tras imprimirse varios hitos de progreso, y el prompt generado y usado por el LLM, se aportan los prints del resultado obtenido por párrafos y el resultado completo.

DeepSeek dió una función para grabar el resultado, que no hemos implementado finalmente.

In [ ]:
def main():
    """Función principal para generar el diario"""

    # DS - Configuración - Clave original borrada por motivos de seguridad

    OPENAI_API_KEY = "You_are_right_to_be_stunned"  # O usar variable de entorno

    # Path del fichero Excel Ejemplo_Diarios.xlsx - Este es el original de mi Colab

    EXCEL_PATH = "/content/drive/MyDrive/Practica_Usos_IA_25/Practica/Ejemplo_Diarios.xlsx"

    # DS - Fecha para el diario (puede ser pasado, presente o futuro)
    # dia_diario = datetime.now()  # Hoy
    # DS - dia_diario = datetime(2024, 11, 12)  # Fecha específica

    dia_diario = datetime(2025, 12, 18)

    print(f"📅 Generando Diario de Bolsa para {dia_diario.strftime('%d/%m/%Y')}")
    print("=" * 60)

    # DS - Inicializar generador
    generator = DiarioGenerator(OPENAI_API_KEY, EXCEL_PATH)

    # DS - Generar diario
    resultado = generator.generate_diario(dia_diario)

    # DS - Mostrar resultados
    print("\n✅ DIARIO GENERADO EXITOSAMENTE")
    print(f"📊 Fecha datos: {resultado['fecha_datos'].strftime('%d/%m/%Y')}")
    print(f"📰 Fecha publicación: {resultado['fecha_publicacion'].strftime('%d/%m/%Y')}")

    print("\n" + "=" * 60)
    print("📝 COMENTARIO DE MERCADO:")
    print("=" * 60)

    print(f"\n📈 PÁRRAFO 1 (Mercados):")
    print(resultado['comentario']['parrafo_1'])

    print(f"\n🌍 PÁRRAFO 2 (Economía):")
    print(resultado['comentario']['parrafo_2'])

    print(f"\n🇪🇸 PÁRRAFO 3 (IBEX):")
    print(resultado['comentario']['parrafo_3'])


    print(f"\nTexto Completo:")
    print(resultado['comentario']['texto_completo'])

    # Opcional: Guardar en archivo
    # save_to_file(resultado, f"diario_{dia_diario.strftime('%Y%m%d')}.json")

    return resultado

'''

def save_to_file(resultado, filename):
    """Guardar resultado en archivo JSON"""
    import json
    import pandas as pd # Import pandas here

    # DS - Convertir fechas y objetos no serializables a string para JSON
    def serialize_value(v):
        if isinstance(v, datetime):
            return v.strftime('%Y-%m-%d %H:%M:%S') # More detailed datetime format
        elif isinstance(v, pd.Series):
            if not v.empty and len(v) == 1:
                return v.item() # Get scalar from single-element Series
            return v.tolist() if not v.empty else None # Convert multi-element Series to list, or empty to None
        elif pd.isna(v): # Handle numpy.nan from pandas
            return None
        return v

    resultado_serializable = {
        'fecha_publicacion': resultado['fecha_publicacion'].strftime('%Y-%m-%d'),
        'fecha_datos': resultado['fecha_datos'].strftime('%Y-%m-%d'),
        'comentario': resultado['comentario'],
        'datos_brutos': {
            k: serialize_value(v)
            for k, v in resultado['datos_brutos'].items()
            if k != 'news'  # DS - Simplificar para ejemplo, news contains complex objects like datetime objects from news article
        }
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(resultado_serializable, f, ensure_ascii=False, indent=2)

    print(f"\n💾 Resultado guardado en {filename}")

    '''

if __name__ == "__main__":
    # DS - Ejecutar
    main()


📅 Generando Diario de Bolsa para 18/12/2025
🔍 Obteniendo datos para 2025-12-18...
  📊 Procesando EuroStoxx50...


ERROR:yfinance:$SX8.DE: possibly delisted; no timezone found
ERROR:yfinance:$EXX3.DE: possibly delisted; no price data found  (1d 2025-01-01 -> 2025-12-18)
ERROR:yfinance:$EXXP.DE: possibly delisted; no timezone found
ERROR:yfinance:$EXXH.DE: possibly delisted; no timezone found


    ✅ EuroStoxx50: 5681.67 (-0.63%)
  📊 Procesando Nasdaq100...
    ✅ Nasdaq100: 24647.61 (-1.93%)
  📊 Procesando S&P500...
    ✅ S&P500: 6721.43 (-1.16%)
  📊 Procesando DAX...
    ✅ DAX: 23960.59 (-0.48%)
  📊 Procesando CAC40...
    ✅ CAC40: 8086.05 (-0.25%)
  📊 Procesando FTSE100...
    ✅ FTSE100: 9774.30 (+0.92%)
  📊 Procesando IBEX35...
    ✅ IBEX35: 16938.20 (+0.10%)
  📊 Procesando sector Bancos (ticker: EXX1.DE)...
    ✅ Bancos: 25.23 (+0.76%)
  📊 Procesando sector Tecnología (ticker: SX8.DE)...
    ⚠️  Sin datos para Tecnología
  📊 Procesando sector Utilities (ticker: EXX5.DE)...
    ✅ Utilities: 86.83 (+0.37%)
  📊 Procesando sector Energía (ticker: EXX3.DE)...
    ⚠️  Sin datos para Energía
  📊 Procesando sector Automoción (ticker: EXX7.DE)...
    ✅ Automoción: 26.96 (-1.50%)
  📊 Procesando sector Farmacia (ticker: EXXP.DE)...
    ⚠️  Sin datos para Farmacia
  📊 Procesando sector Telecoms (ticker: EXS1.DE)...
    ✅ Telecoms: 198.28 (-0.55%)
  📊 Procesando sector Construcción (t

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['PUI.MC']: YFTzMissingError('possibly delisted; no timezone found')


{'Santander': {'precio': 9.87, 'variacion': 0.55, 'volumen': 23541623}, 'BBVA': {'precio': 19.19, 'variacion': -0.6, 'volumen': 10648447}, 'Telefónica': {'precio': 3.49, 'variacion': 0.93, 'volumen': 16660254}, 'Inditex': {'precio': 54.8, 'variacion': 0.44, 'volumen': 2454211}, 'Iberdrola': {'precio': 17.94, 'variacion': -0.17, 'volumen': 8922339}, 'Repsol': {'precio': 15.45, 'variacion': 0.78, 'volumen': 4138518}, 'Ferrovial': {'precio': 56.78, 'variacion': -0.53, 'volumen': 700936}, 'ACS': {'precio': 83.3, 'variacion': -2.29, 'volumen': 338704}, 'Mapfre': {'precio': 4.23, 'variacion': 1.49, 'volumen': 4124928}, 'Grifols': {'precio': 10.68, 'variacion': -1.2, 'volumen': 1092180}, 'Aena': {'precio': 23.47, 'variacion': 0.0, 'volumen': 1118143}, 'IAG': {'precio': 4.79, 'variacion': 0.4, 'volumen': 5475529}, 'Indra': {'precio': 45.76, 'variacion': -1.08, 'volumen': 817459}, 'Sabadell': {'precio': 3.36, 'variacion': 0.48, 'volumen': 16736625}, 'CaixaBank': {'precio': 10.31, 'variacion': 1